# Amparo -- M3: RAG ingenuo

Construye y consulta el RAG ingenuo (alcance S07: `ingest -> chunk -> embed ->
store` offline, `retrieve -> augment -> generate` online) sobre el corpus de
normas colombianas versionado en `data/corpus/normas/`.

Toda la logica vive en `tools/rag/` (repo principal, no en este notebook) --
aca solo se clona el repo, se instala el stack pesado y se corren las fases,
igual que hacen `baseline_finetune.ipynb` (M1) y `evaluacion.ipynb` (M2).

**Requiere GPU** (Entorno de ejecucion -> Cambiar tipo de entorno -> T4 o
superior). Las decisiones de diseno estan en `docs/m3_decisiones_rag.md`.

**Requiere tambien que `tools/rag/` este publicado** en la rama que se clona
abajo: este notebook corre contra el repo remoto, no contra tu maquina. Si el
codigo todavia esta solo en local, cambia `RAMA` por la rama donde este.

Fases: verificar corpus -> construir indice -> consulta de prueba -> valvula de
escape -> calibrar el umbral -> corrida sobre `eval_set.json`, persistida en Drive.

In [1]:
# Rama del repo a clonar. Cambiala si el codigo del RAG todavia no esta en
# main (por ejemplo, mientras vive en una rama de PR como "m3-rag-ingenuo").
RAMA = "main"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

Cloning into 'Amparo'...
remote: Enumerating objects: 275, done.
remote: Counting objects: 100% (275/275), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 275 (delta 120), reused 245 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (275/275), 2.28 MiB | 21.21 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/Amparo


In [2]:
# Verifica que la rama clonada traiga lo que este notebook necesita, ANTES de
# instalar nada ni cargar modelos. Sin esta celda, la falta de tools/rag/ se
# manifiesta recien en la Fase 1 como "ModuleNotFoundError: No module named
# 'tools.rag'", que no dice el motivo real: que el codigo no esta publicado en
# la rama clonada (el clone solo trae lo que hay en el remoto).
import os, subprocess, sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/pipeline.py",
    "tools/rag/corpus.py",
    "tools/evaluation/generation.py",
    "data/corpus/normas",
    "data/eval_set.json",
]
faltan = [ruta for ruta in REQUERIDOS if not os.path.exists(ruta)]

if faltan:
    rama = subprocess.run(
        ["git", "branch", "--show-current"], capture_output=True, text=True
    ).stdout.strip()
    raise SystemExit(
        f"Falta en la rama '{rama}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG todavia "
        "esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA en la "
        "celda anterior por la rama donde este publicado."
    )

print(f"Repo OK en {os.getcwd()} (rama "
      f"{subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip()})")

Repo OK en /content/Amparo (rama main)


In [3]:
# Dependencias livianas del repo (incluye faiss-cpu, el backend del indice)
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 129.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 17.0 MB/s eta 0:00:00


In [4]:
# Stack pesado -- igual que en M1/M2: se instala aqui y NO en requirements.txt,
# para no arriesgar reemplazar el build de PyTorch con CUDA que Colab ya trae.
# transformers alcanza para el modelo de embeddings (e5) y para la generacion;
# peft/bitsandbytes solo hacen falta si se genera con el adaptador LoRA de M1.
!pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 148.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.9 MB/s eta 0:00:00


## Fase 1 -- Verificar el corpus

El corpus esta versionado en el repo, asi que no hay nada que descargar. Esta
celda valida el manifiesto: que cada norma en alcance exista, que su frontmatter
traiga la metadata citable (`identifier`, `rank`, `status`, `source`) y que las 9
categorias objetivo queden cubiertas. Si algo falta, falla **aca** y no despues
de haber cargado el modelo.

In [5]:
from tools.rag import config, corpus

manifest = corpus.to_ingest_manifest()   # valida al construir

print(f"{len(corpus.archivos_del_corpus())} normas descargadas | "
      f"{len(manifest)} en alcance | {len(corpus.FUERA_DE_ALCANCE)} fuera de alcance")
print(f"Categorias cubiertas: {len(corpus.categorias_cubiertas())} de {len(corpus.CATEGORIAS_OBJETIVO)}\n")
for entrada in manifest:
    print("  " + entrada["fuente"])

17 normas descargadas | 10 en alcance | 7 fuera de alcance
Categorias cubiertas: 9 de 9

  Constitucion Politica de 1991
  Ley 1437 de 2011 (CPACA)
  Decreto 2591 de 1991 (Reglamentacion de la accion de tutela)
  Ley 100 de 1993 (Sistema de Seguridad Social Integral)
  Decreto 2663 de 1950 (Codigo Sustantivo del Trabajo)
  Ley 820 de 2003 (Regimen de arrendamiento de vivienda urbana)
  Ley 1266 de 2008 (Habeas data financiero)
  Ley 1480 de 2011 (Estatuto del Consumidor)
  Ley 769 de 2002 (Codigo Nacional de Transito)
  Ley 1564 de 2012 (Codigo General del Proceso)


## Fase 2 -- Construir el indice

`ingest -> chunk -> embed -> store`. Los embeddings son de
`intfloat/multilingual-e5-base` con prefijo `passage:`; el indice es FAISS
(`IndexFlatIP` sobre vectores normalizados = coseno exacto).

Con ~3.400 chunks tarda unos minutos en GPU. El indice queda en `artifacts/`.

In [6]:
from tools.rag import pipeline

store = pipeline.build_index()
print(f"\nChunks indexados: {len(store)}")

[1/4] ingest...
[2/4] chunk...
      10 documentos -> 3425 chunks
[3/4] embed (intfloat/multilingual-e5-base)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      250/3425 (7%) -- ETA ~8.3 min
      500/3425 (15%) -- ETA ~4.0 min
      750/3425 (22%) -- ETA ~2.5 min
      1000/3425 (29%) -- ETA ~1.8 min
      1250/3425 (36%) -- ETA ~1.3 min
      1500/3425 (44%) -- ETA ~1.0 min
      1750/3425 (51%) -- ETA ~0.8 min
      2000/3425 (58%) -- ETA ~0.6 min
      2250/3425 (66%) -- ETA ~0.5 min
      2500/3425 (73%) -- ETA ~0.3 min
      2750/3425 (80%) -- ETA ~0.2 min
      3000/3425 (88%) -- ETA ~0.1 min
      3250/3425 (95%) -- ETA ~0.1 min
      3425/3425 (100%) -- ETA ~0.0 min
[4/4] store (faiss)...
      indice -> /content/Amparo/artifacts/rag_index.faiss
Listo: 3425 chunks indexados.

Chunks indexados: 3425


In [7]:
# Copia del indice a Drive, para no tener que reconstruirlo en cada sesion
# (mismo patron que M1/M2 con el adaptador y las salidas de evaluacion).
from google.colab import drive
import pathlib, shutil

drive.mount("/content/drive")
destino = pathlib.Path(config.DRIVE_ROOT) / "rag"
destino.mkdir(parents=True, exist_ok=True)
for archivo in (config.FAISS_INDEX_PATH, config.FAISS_METADATA_PATH):
    shutil.copy(archivo, destino / archivo.name)
print(f"Indice copiado a {destino}")

# Para retomar en una sesion nueva sin reindexar, en vez de build_index():
#   for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
#       shutil.copy(destino / nombre, config.ARTIFACTS_DIR / nombre)
#   store = pipeline.load_index()

Mounted at /content/drive
Indice copiado a /content/drive/MyDrive/Colab Notebooks/Amparo/rag


## Fase 3 -- Consulta de prueba

`retrieve -> augment -> generate`. El modelo se carga una sola vez y se reusa:
cargarlo por consulta son 7B de pesos cada vez.

`USE_LORA=True` genera con el adaptador de M1 encima. Comparar las dos
configuraciones ("RAG sobre base" vs. "RAG sobre fine-tuneado") es parte de lo
que M3 tiene que reportar: dice si el fine-tuning sigue aportando cuando el
modelo ya tiene fuentes verificadas en el contexto.

In [8]:
USE_LORA = False   # cambiar a True para generar con el adaptador de M1

model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo cargado.


In [9]:
consulta = "Me despidieron sin pagarme la liquidacion, que puedo hacer?"

resultado = pipeline.answer_query(consulta, store, use_lora=USE_LORA, model_bundle=model_bundle)

print(resultado["response"])
print("\n--- Fuentes recuperadas ---")
for c in resultado["retrieved_chunks"]:
    print(f"  [{c['score']:.3f}] {c['cita']}")

Para resolver tu situación, es importante considerar varios aspectos legales. Primero, debes revisar si tienes un contrato de trabajo que especifique cómo se debe realizar la liquidación salarial. Si no existe tal contrato, puedes:

1. **Consultar al sindicato**: Si estás afiliado a un sindicato, puedes buscar ayuda allí, ya que la ley colombiana requiere que el liquidador designado por los afiliados o por el Juez aplique los fondos existentes para pagar las deudas del sindicato, incluyendo las salariales.

2. **Presentar una denuncia ante la Inspectora Regional del Trabajo**: Esta entidad puede intervenir en casos de despidos injustificados y falta de pago de liquidaciones.

3. **Demandar judicialmente**: Puedes presentar una demanda ante un juez laboral para exigir el pago de la liquidación adeudada. Según el Decreto 2663 de 1950, el liquidador debe aplicar los fondos para pagar las deudas del sindicato, federación o confederación, incluyendo las salariales.

4. **Reunir pruebas**: E

## Fase 4 -- Valvula de escape

La prueba que el checklist de M3 pide: una pregunta **fuera del corpus** debe
producir la frase de escape, no una respuesta inventada. Sin esto, el RAG no
resuelve el hallazgo de M2 ("el modelo cede y cita bajo presion aunque no tenga
de donde verificar") -- solo lo esconde mejor detras de contexto real.

In [10]:
from tools.rag import prompt_template

fuera_de_corpus = [
    "Que dice la ley de patentes mineras sobre las regalias?",
    "Cual es el plazo para apelar una multa de transito en Argentina?",
]

for consulta in fuera_de_corpus:
    r = pipeline.answer_query(consulta, store, use_lora=USE_LORA, model_bundle=model_bundle)
    activo = prompt_template.RESPUESTA_SIN_CONTEXTO.lower()[:40] in r["response"].lower()
    print(f"[{'OK' if activo else 'REVISAR'}] chunks={r['n_retrieved']} | {consulta}")
    print(f"    {r['response'][:200]}\n")

[REVISAR] chunks=4 | Que dice la ley de patentes mineras sobre las regalias?
    No tengo información verificada sobre esto en mi base de conocimiento.

[OK] chunks=5 | Cual es el plazo para apelar una multa de transito en Argentina?
    No tengo informacion verificada sobre esto en mi base de conocimiento.



## Fase 5 -- Calibrar el umbral de la valvula de escape

`RETRIEVAL_MIN_SCORE` esta hoy en 0.80 **sin calibrar** (los cosenos de e5 son
poco dispersos: dos textos sin relacion suelen dar ~0.70-0.75, asi que un piso
bajo dejaria pasar cualquier cosa y la valvula nunca se activaria). Esta celda
recupera sin piso y compara la distribucion de scores de preguntas **con**
cobertura (gold) contra las de **sin** cobertura, para elegir el corte con datos
y anotarlo en la seccion 5 de `docs/m3_decisiones_rag.md`.

In [11]:
import statistics
from tools.evaluation import eval_set
from tools.rag import retrieve

registros = eval_set.load_eval_set()
gold = eval_set.gold_examples(registros)[:20]

def top1(consulta):
    r = retrieve.retrieve(consulta, store, min_score=None)
    return r[0].score if r else 0.0

con_cobertura = [top1(g["messages"][1]["content"]) for g in gold]
sin_cobertura = [top1(q) for q in fuera_de_corpus]

print(f"CON cobertura (n={len(con_cobertura)}): min {min(con_cobertura):.3f} | "
      f"mediana {statistics.median(con_cobertura):.3f} | max {max(con_cobertura):.3f}")
print(f"SIN cobertura (n={len(sin_cobertura)}): min {min(sin_cobertura):.3f} | "
      f"max {max(sin_cobertura):.3f}")
print(f"\nUmbral actual: {config.RETRIEVAL_MIN_SCORE}")
print("Un buen corte queda por encima del max de SIN cobertura y por debajo del min de CON cobertura.")
print("Si los dos rangos se solapan, el umbral solo no alcanza: anotalo como consulta fallida.")

CON cobertura (n=20): min 0.825 | mediana 0.841 | max 0.879
SIN cobertura (n=2): min 0.840 | max 0.854

Umbral actual: 0.82
Un buen corte queda por encima del max de SIN cobertura y por debajo del min de CON cobertura.
Si los dos rangos se solapan, el umbral solo no alcanza: anotalo como consulta fallida.


## Fase 6 -- Corrida sobre el eval set

Corre el RAG sobre `data/eval_set.json` (gold + adversariales) y guarda la
salida en formato Ragas (`question` / `answer` / `contexts` / `ground_truth`),
mas la evidencia de retrieval para auditoria.

**La evaluacion en si no es parte de M3**: este archivo es el insumo para quien
continue con el RAG avanzado y la evaluacion con Ragas y el harness.

In [12]:
import json, time
from datetime import datetime, timezone

registros = eval_set.load_eval_set()
salida = []
inicio = time.perf_counter()

for i, registro in enumerate(registros, start=1):
    consulta = registro["messages"][1]["content"]
    resultado = pipeline.answer_query(consulta, store, use_lora=USE_LORA, model_bundle=model_bundle)
    salida.append(pipeline.to_eval_record(resultado, registro))
    if i % 10 == 0 or i == len(registros):
        transcurrido = time.perf_counter() - inicio
        eta = (transcurrido / i) * (len(registros) - i) / 60
        print(f"{i}/{len(registros)} -- {transcurrido/i:.1f}s/consulta, ETA ~{eta:.1f} min")

etiqueta = "lora" if USE_LORA else "base"
sello = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
nombre = f"rag_eval_{etiqueta}_{sello}.jsonl"

for ruta in (config.ARTIFACTS_DIR / nombre, destino / nombre):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        for registro in salida:
            f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"Guardado: {ruta}")

10/56 -- 10.7s/consulta, ETA ~8.2 min
20/56 -- 8.1s/consulta, ETA ~4.9 min
30/56 -- 7.9s/consulta, ETA ~3.4 min
40/56 -- 8.4s/consulta, ETA ~2.2 min
50/56 -- 8.2s/consulta, ETA ~0.8 min
56/56 -- 7.9s/consulta, ETA ~0.0 min
Guardado: /content/Amparo/artifacts/rag_eval_base_2026-09-25_171025.jsonl
Guardado: /content/drive/MyDrive/Colab Notebooks/Amparo/rag/rag_eval_base_2026-09-25_171025.jsonl


In [13]:
# Resumen de sanidad de la corrida (no es la evaluacion)
sin_contexto = [r for r in salida if r["n_retrieved"] == 0]
adversariales = [r for r in salida if r["tipo"] == "adversarial"]

print(f"Consultas: {len(salida)} | sin contexto recuperado: {len(sin_contexto)}")
print(f"Adversariales: {len(adversariales)}, sin contexto: "
      f"{sum(1 for r in adversariales if r['n_retrieved'] == 0)}")
print(f"Chunks por consulta (promedio): {sum(r['n_retrieved'] for r in salida)/len(salida):.1f}")

Consultas: 56 | sin contexto recuperado: 1
Adversariales: 6, sin contexto: 1
Chunks por consulta (promedio): 4.6


## Que sigue (fuera del alcance de M3)

1. **Anotar las consultas fallidas** en la tabla de la seccion 8 de
   `docs/m3_decisiones_rag.md`, con la etapa responsable (ingest / chunk / embed
   / retrieve / generate). Es el insumo que decide, con datos, si hace falta S08
   (hybrid search, reranking, query transformation).
2. **Evaluacion con Ragas y con el harness de `tools/evaluation/`** sobre el
   `.jsonl` que dejo la fase 6.
3. **Comparacion `USE_LORA=False` vs `True`**: correr la fase 6 dos veces y
   contrastar.

Ninguna de las tres se implementa aca: M3 entrega el RAG ingenuo funcionando y
su salida en el formato que esas tres necesitan.